In [1]:
import pandas as pd
import numpy as np
from xbbg import blp
from pathlib import Path
import time
import json

# Load master ticker list
master = pd.read_parquet('../data/raw/master_tickers.parquet')
tickers = master['ticker'].tolist()

OUT_DIR = Path('../data/raw/equities')
PROGRESS_PATH = Path('../data/raw/price_progress.json')

START = '2009-07-01'   # 6-month buffer before 2010-01 for momentum lookbacks
END = '2025-12-31'

FIELDS = [
    'PX_LAST', 'PX_OPEN', 'PX_HIGH', 'PX_LOW',
    'PX_VOLUME', 'EQY_SH_OUT', 'CUR_MKT_CAP',
    'TOT_RETURN_INDEX_GROSS_DVDS'
]

CHUNK_SIZE = 20  # tickers per batch

# Split into chunks
chunks = [tickers[i:i+CHUNK_SIZE] for i in range(0, len(tickers), CHUNK_SIZE)]

print(f"Tickers: {len(tickers)}")
print(f"Fields: {len(FIELDS)}")
print(f"Date range: {START} to {END}")
print(f"Chunks: {len(chunks)} (of {CHUNK_SIZE} tickers each)")

Tickers: 641
Fields: 8
Date range: 2009-07-01 to 2025-12-31
Chunks: 33 (of 20 tickers each)


In [2]:
async def pull_chunk(batch, chunk_idx):
    """Pull daily price data for a batch of tickers. Returns DataFrame or None."""
    for attempt in range(3):
        try:
            df = await blp.abdh(
                tickers=batch,
                flds=FIELDS,
                start_date=START,
                end_date=END,
                adjust='-'
            )
            # Convert to pandas if needed
            if hasattr(df, 'to_pandas'):
                df = df.to_pandas()
           
            if df is not None and len(df) > 0:
                return df
            else:
                print(f"  Attempt {attempt+1}: empty result, retrying...")
                time.sleep(10)
        except Exception as e:
            print(f"  Attempt {attempt+1} error: {e}")
            time.sleep(15)
    return None

In [3]:
# Resume support
completed_chunks = set()
if PROGRESS_PATH.exists():
    with open(PROGRESS_PATH) as f:
        progress = json.load(f)
        completed_chunks = set(progress.get('completed_chunks', []))
        print(f"Resuming: {len(completed_chunks)} chunks already done")

total_rows = 0
failed_chunks = []

for chunk_idx, batch in enumerate(chunks):
    if str(chunk_idx) in completed_chunks:
        print(f"[{chunk_idx+1}/{len(chunks)}] Skipping (already done)")
        continue
   
    print(f"[{chunk_idx+1}/{len(chunks)}] Pulling {len(batch)} tickers...", end=' ')
   
    df = await pull_chunk(batch, chunk_idx)
   
    if df is not None and len(df) > 0:
        out_path = OUT_DIR / f'chunk_{chunk_idx:03d}.parquet'
        df.to_parquet(out_path)
        completed_chunks.add(str(chunk_idx))
        print(f"OK — {len(df)} rows saved")
        total_rows += len(df)
    else:
        print(f"FAILED")
        failed_chunks.append(chunk_idx)
   
    # Save progress after each chunk
    with open(PROGRESS_PATH, 'w') as f:
        json.dump({'completed_chunks': list(completed_chunks)}, f)
   
    time.sleep(2)

print(f"\nDone! {len(completed_chunks)}/{len(chunks)} chunks complete")
print(f"Total rows saved: {total_rows:,}")
if failed_chunks:
    print(f"Failed chunks: {failed_chunks}")
else:
    print("No failures!")

[1/33] Pulling 20 tickers... OK — 188008 rows saved
[2/33] Pulling 20 tickers... OK — 426848 rows saved
[3/33] Pulling 20 tickers... OK — 361640 rows saved
[4/33] Pulling 20 tickers... OK — 439176 rows saved
[5/33] Pulling 20 tickers... OK — 531536 rows saved
[6/33] Pulling 20 tickers... OK — 468448 rows saved
[7/33] Pulling 20 tickers... OK — 449552 rows saved
[8/33] Pulling 20 tickers... OK — 486864 rows saved
[9/33] Pulling 20 tickers... OK — 409184 rows saved
[10/33] Pulling 20 tickers... OK — 495536 rows saved
[11/33] Pulling 20 tickers... OK — 471016 rows saved
[12/33] Pulling 20 tickers... OK — 437120 rows saved
[13/33] Pulling 20 tickers... OK — 541704 rows saved
[14/33] Pulling 20 tickers... OK — 524232 rows saved
[15/33] Pulling 20 tickers... OK — 542856 rows saved
[16/33] Pulling 20 tickers... OK — 460832 rows saved
[17/33] Pulling 20 tickers... OK — 526672 rows saved
[18/33] Pulling 20 tickers... OK — 511800 rows saved
[19/33] Pulling 20 tickers... OK — 498416 rows saved
[2

In [4]:
# FTSE 250 benchmark
print("Pulling benchmark (MCX Index)...", end=' ')
bench = await blp.abdh(
    tickers='MCX Index',
    flds=['PX_LAST', 'TOT_RETURN_INDEX_GROSS_DVDS'],
    start_date=START,
    end_date=END
)
if hasattr(bench, 'to_pandas'):
    bench = bench.to_pandas()
bench.to_parquet('../data/raw/benchmark.parquet')
print(f"OK — {len(bench)} rows")

# SONIA overnight rate
print("Pulling SONIA (SONIO/N Index)...", end=' ')
sonia = await blp.abdh(
    tickers='SONIO/N Index',
    flds=['PX_LAST'],
    start_date=START,
    end_date=END
)
if hasattr(sonia, 'to_pandas'):
    sonia = sonia.to_pandas()
sonia.to_parquet('../data/raw/sonia.parquet')
print(f"OK — {len(sonia)} rows")

print("\nBenchmark preview:")
print(bench.tail(3))
print("\nSONIA preview:")
print(sonia.tail(3))

Pulling benchmark (MCX Index)... OK — 8340 rows
Pulling SONIA (SONIO/N Index)... OK — 4170 rows

Benchmark preview:
         ticker        date                        field       value
8337  MCX Index  2025-12-30  TOT_RETURN_INDEX_GROSS_DVDS  35538.2353
8338  MCX Index  2025-12-31                      PX_LAST  22470.3800
8339  MCX Index  2025-12-31  TOT_RETURN_INDEX_GROSS_DVDS  35399.6324

SONIA preview:
             ticker        date    field   value
4167  SONIO/N Index  2025-12-29  PX_LAST  3.7251
4168  SONIO/N Index  2025-12-30  PX_LAST  3.7254
4169  SONIO/N Index  2025-12-31  PX_LAST  3.7257


In [5]:
# Bid-ask data — coverage will be patchy for mid-caps, that's expected
# We also already have PX_HIGH and PX_LOW in the main panel for Corwin-Schultz

print("Pulling bid-ask data for all tickers...")
print("(This may take 30-40 minutes — same chunked approach)")

BA_DIR = Path('../data/raw/bid_ask')
BA_DIR.mkdir(parents=True, exist_ok=True)
BA_PROGRESS = Path('../data/raw/ba_progress.json')

ba_completed = set()
if BA_PROGRESS.exists():
    with open(BA_PROGRESS) as f:
        ba_completed = set(json.load(f).get('completed', []))
        print(f"Resuming: {len(ba_completed)} chunks done")

ba_failed = []

for chunk_idx, batch in enumerate(chunks):
    if str(chunk_idx) in ba_completed:
        print(f"[{chunk_idx+1}/{len(chunks)}] Skipping (done)")
        continue
   
    print(f"[{chunk_idx+1}/{len(chunks)}] Pulling bid-ask...", end=' ')
   
    for attempt in range(3):
        try:
            df = await blp.abdh(
                tickers=batch,
                flds=['BID', 'ASK'],
                start_date='2010-01-01',
                end_date=END,
                adjust='-'
            )
            if hasattr(df, 'to_pandas'):
                df = df.to_pandas()
            if df is not None and len(df) > 0:
                df.to_parquet(BA_DIR / f'ba_chunk_{chunk_idx:03d}.parquet')
                ba_completed.add(str(chunk_idx))
                print(f"OK — {len(df)} rows")
                break
            else:
                print(f"empty, retry {attempt+1}...", end=' ')
                time.sleep(5)
        except Exception as e:
            print(f"error: {e}, retry {attempt+1}...", end=' ')
            time.sleep(10)
    else:
        print("FAILED")
        ba_failed.append(chunk_idx)
   
    with open(BA_PROGRESS, 'w') as f:
        json.dump({'completed': list(ba_completed)}, f)
    time.sleep(2)

print(f"\nBid-ask pull done! {len(ba_completed)}/{len(chunks)} chunks")
if ba_failed:
    print(f"Failed: {ba_failed}")
else:
    print("No failures!")
print("Note: patchy coverage is EXPECTED for mid-caps — Corwin-Schultz is your primary spread estimator")

Pulling bid-ask data for all tickers...
(This may take 30-40 minutes — same chunked approach)
[1/33] Pulling bid-ask... OK — 42752 rows
[2/33] Pulling bid-ask... OK — 102626 rows
[3/33] Pulling bid-ask... OK — 88120 rows
[4/33] Pulling bid-ask... OK — 104284 rows
[5/33] Pulling bid-ask... OK — 128842 rows
[6/33] Pulling bid-ask... OK — 112658 rows
[7/33] Pulling bid-ask... OK — 108244 rows
[8/33] Pulling bid-ask... OK — 112880 rows
[9/33] Pulling bid-ask... OK — 99064 rows
[10/33] Pulling bid-ask... OK — 117362 rows
[11/33] Pulling bid-ask... OK — 112098 rows
[12/33] Pulling bid-ask... OK — 104016 rows
[13/33] Pulling bid-ask... OK — 131654 rows
[14/33] Pulling bid-ask... OK — 126984 rows
[15/33] Pulling bid-ask... OK — 130706 rows
[16/33] Pulling bid-ask... OK — 109760 rows
[17/33] Pulling bid-ask... OK — 127910 rows
[18/33] Pulling bid-ask... OK — 120470 rows
[19/33] Pulling bid-ask... OK — 120492 rows
[20/33] Pulling bid-ask... OK — 125096 rows
[21/33] Pulling bid-ask... OK — 133886